In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.float_format", "{:.3f}".format)

df = pd.read_csv("RIPAoc2020.csv", low_memory=False)

# basic cleaning
df["DATE_OF_STOP"] = pd.to_datetime(df["DATE_OF_STOP"], errors="coerce")
df = df[df["DATE_OF_STOP"].dt.year == 2020].copy()

# ensure stop reason is string
df["REASON_FOR_STOP"] = df["REASON_FOR_STOP"].astype(str).str.upper()

/var/folders/4n/gr221w3n74g83smf7w9pzfy00000gn/T/ipykernel_13695/108664364.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["DATE_OF_STOP"] = pd.to_datetime(df["DATE_OF_STOP"], errors="coerce")


In [2]:
race_cols = {
    "White": "RAE_WHITE",
    "Black": "RAE_BLACK_AFRICAN_AMERICAN",
    "Hispanic": "RAE_HISPANIC_LATINO",
    "Asian": "RAE_ASIAN",
    "Other/Unknown": "RAE_MULTIRACIAL"
}

# ensure numeric
for c in race_cols.values():
    df[c] = pd.to_numeric(df[c], errors="coerce")

def get_race(row):
    for race, col in race_cols.items():
        if row[col] == 1:
            return race
    return np.nan

df["RACE"] = df.apply(get_race, axis=1)
df = df[df["RACE"].notna()].copy()

In [3]:
rs_cols = [
    "RFS_RS_REASON_SUSP",
    "RFS_RS_VIOLENT_CRIME",
    "RFS_RS_DRUG_TRANS",
    "RFS_RS_MATCH_SUSPECT"
]

for c in rs_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["MOVING"] = df["REASON_FOR_STOP"].str.contains("MOVING", na=False)
df["EQUIPMENT"] = df["REASON_FOR_STOP"].str.contains("EQUIP", na=False)

df["INVESTIGATIVE"] = (
    (df["RFS_RS_REASON_SUSP"] == 1) |
    (df["RFS_RS_VIOLENT_CRIME"] == 1) |
    (df["RFS_RS_DRUG_TRANS"] == 1) |
    (df["RFS_RS_MATCH_SUSPECT"] == 1)
)

In [4]:
table1 = df.groupby("RACE").agg(
    total_stops=("RACE", "size"),
    moving=("MOVING", "mean"),
    equipment=("EQUIPMENT", "mean"),
    investigative=("INVESTIGATIVE", "mean")
)

# as percent
table1[["moving", "equipment", "investigative"]] *= 100

# percent of all stops
table1["pct_stops"] = table1["total_stops"] / len(df) * 100

# reorder
table1 = table1[[
    "total_stops",
    "pct_stops",
    "moving",
    "equipment",
    "investigative"
]]

table1.columns = [
    "Total Stops",
    "% of Stops",
    "Moving (%)",
    "Equipment (%)",
    "Investigative (%)"
]

table1

,Total Stops,% of Stops,Moving (%),Equipment (%),Investigative (%)
RACE,,,,,
Asian,2520,6.785,0.000,0.000,2.421
Black,1585,4.268,0.000,0.000,8.833
Hispanic,13612,36.651,0.000,0.000,4.834
White,19423,52.297,0.000,0.000,5.241


In [5]:
total_row = pd.DataFrame({
    "Total Stops": [len(df)],
    "% of Stops": [100.0],
    "Moving (%)": [df["MOVING"].mean() * 100],
    "Equipment (%)": [df["EQUIPMENT"].mean() * 100],
    "Investigative (%)": [df["INVESTIGATIVE"].mean() * 100],
}, index=["Total"])

table1_final = pd.concat([table1, total_row])

table1_final

,Total Stops,% of Stops,Moving (%),Equipment (%),Investigative (%)
Asian,2520,6.785,0.000,0.000,2.421
Black,1585,4.268,0.000,0.000,8.833
Hispanic,13612,36.651,0.000,0.000,4.834
White,19423,52.297,0.000,0.000,5.241
Total,37140,100.000,0.000,0.000,5.054


In [14]:
df_vod = df.copy()
df_vod["TIME_OF_STOP"] = pd.to_numeric(df_vod["TIME_OF_STOP"], errors="coerce")

df_vod = df_vod[df_vod["TIME_OF_STOP"].notna()].copy()

df_vod["HOUR"] = (df_vod["TIME_OF_STOP"] // 100).astype(int)
df_vod["MINUTE"] = (df_vod["TIME_OF_STOP"] % 100).astype(int)

df_vod = df_vod[(df_vod["MINUTE"] < 60) & (df_vod["HOUR"] <= 23)].copy()

In [15]:
df_vod = df_vod[
    (df_vod["HOUR"] >= 17) &
    (df_vod["HOUR"] <= 20)
].copy()

In [16]:
df_vod["BLACK"] = pd.to_numeric(df_vod["RAE_BLACK_AFRICAN_AMERICAN"], errors="coerce").fillna(0)
df_vod["HISP"] = pd.to_numeric(df_vod["RAE_HISPANIC_LATINO"], errors="coerce").fillna(0)

df_vod["MINORITY"] = ((df_vod["BLACK"] == 1) | (df_vod["HISP"] == 1)).astype(int)

In [19]:
df_vod["TIME_BIN"] = df_vod["HOUR"] * 4 + (df_vod["MINUTE"] // 15)

# Veil of Darkness approximation
df_vod["DAYLIGHT"] = (df_vod["HOUR"] < 19).astype(int)

In [20]:
import statsmodels.formula.api as smf

m1 = smf.ols("MINORITY ~ DAYLIGHT + C(TIME_BIN)", data=df_vod).fit(cov_type="HC1")
m2 = smf.ols("BLACK ~ DAYLIGHT + C(TIME_BIN)", data=df_vod).fit(cov_type="HC1")
m3 = smf.ols("HISP ~ DAYLIGHT + C(TIME_BIN)", data=df_vod).fit(cov_type="HC1")

print(len(df_vod))

5212


In [ ]:
# Model outputs
results_table = pd.DataFrame({
    "Model": ["Black or Hispanic", "Black", "Hispanic"],
    "Daylight Coef": [
        m1.params["DAYLIGHT"],
        m2.params["DAYLIGHT"],
        m3.params["DAYLIGHT"]
    ],
    "Std. Error": [
        m1.bse["DAYLIGHT"],
        m2.bse["DAYLIGHT"],
        m3.bse["DAYLIGHT"]
    ],
    "t-Statistic": [
        m1.tvalues["DAYLIGHT"],
        m2.tvalues["DAYLIGHT"],
        m3.tvalues["DAYLIGHT"]
    ],
    "P-Value": [
        m1.pvalues["DAYLIGHT"],
        m2.pvalues["DAYLIGHT"],
        m3.pvalues["DAYLIGHT"]
    ],
    "Observations": [
        int(m1.nobs),
        int(m2.nobs),
        int(m3.nobs)
    ]
})

results_table

,Model,Daylight Coef,Std. Error,t-Statistic,P-Value,Observations
0,Black or Hispanic,0.095,0.024,3.959,0.000,5212
1,Black,0.005,0.010,0.477,0.633,5212
2,Hispanic,0.089,0.024,3.778,0.000,5212
